In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent  # go up from dev_notebooks/
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repo root:", repo_root)

In [ ]:
import time
from datetime import datetime

In [ ]:
import torch
import dac
import torch.optim as optim

#for tensorboard
import os
from torch.utils.tensorboard import SummaryWriter


from torch.utils.data import DataLoader
from utils.io import save_checkpoint, save_run_config

from core.rnndac_dataset import RNNDACLatentDataset, LatentDatasetConfig
from core.rnndac_model import GRUModelConfig, RNNDACModel#, RNNDACModelNoCascade

In [ ]:
#Prepare DAC model
dac_model_path = dac.utils.download(model_type="44khz") 
print(f'Model is stored here: {dac_model_path}')
dac_model = dac.DAC.load(dac_model_path)



In [ ]:
#PARAMETERS

In [ ]:
DEVICE = "cpu" #"cuda" if torch.cuda.is_available() else "cpu"
hf_dataset_path = "/slowdisk/data/DAC/pitchglidesATriangle5octaves/Aglides/hf_dataset"
savename="inftest"


params = {
    
    
    "n_steps": 2500,
    "print_every": 250,
    "validate_every": 250,
    "val_batches": 4,            # choose a small number like 2–8

    "checkpoint_every": 500,

    "cascade_mode": "teacher",   # "teacher", "soft", "hard"
    "cond_injection": "film",    # "concat", "film"
}


In [ ]:
#Prepare output directories
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
savemodeldir = os.path.join(os.getcwd(), 'output')
out_dir = os.path.join(savemodeldir, run_timestamp + "_" + savename)
os.makedirs(out_dir, exist_ok=True)
os.makedirs(f"{out_dir}/checkpoints", exist_ok=True)
checkpoint_dir = os.path.join(out_dir, "checkpoints")
os.makedirs(f"{out_dir}/config", exist_ok=True)
config_dir = out_dir + "/config/"

print("out_dir:", out_dir)
print("checkpoint_dir :", checkpoint_dir )
print("config_dir:", config_dir)

#Prepare tensorboard
log_dir = os.path.join(out_dir, "tensorboard")
writer = SummaryWriter(log_dir=log_dir)
print("TensorBoard log dir:", log_dir)



# # ---- Create Output Folders ----
# if resume_checkpoint != None:
#     out_dir = resume_checkpoint
# else:
#     run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
#     out_dir = os.path.join(params['savemodeldir'], run_timestamp + "_"+savename)
#     os.makedirs(out_dir, exist_ok=True)

#     os.makedirs(out_dir, exist_ok=True)
#     os.makedirs(f"{out_dir}/checkpoints", exist_ok=True)
#     os.makedirs(f"{out_dir}/tensorboard", exist_ok=True)

# save_run_config(f"{out_dir}/config_v2.pt", params=None, model_config=model_config, data_config=data_config)
# print(f"wrote {out_dir}/config_v2.pt")

In [ ]:
# Data set, Validation set, and Model configuration 


data_cfg = LatentDatasetConfig(
    dataset_path=hf_dataset_path,
    sequence_length=128,
    n_q=9,
    clamp_val=15,
    files_per_sequence=4,
)

validation_cfg = LatentDatasetConfig(
    dataset_path=hf_dataset_path,
    sequence_length=128,
    n_q=9,
    clamp_val=15,
    files_per_sequence=1,
)


model_cfg = GRUModelConfig(
    n_q=9,
    codebook_size=1024,
    codebook_dim=8,
    input_size=72,
    cond_size=1,
    hidden_size=128,
    num_layers=3,
    cond_injection=params["cond_injection"],   # or "concat"
)


save_run_config(
    output_dir=checkpoint_dir,
    params=params,
    model_config=model_cfg,
    data_config=data_cfg,
)


In [ ]:
#Create the RNN model
#model = RNNDACModel(cfg, codebook_vectors)
model = RNNDACModel(model_cfg, dac_model=dac_model)
#model = RNNDACModelNoCascade(model_cfg, dac_model=dac_model)
model = model.to(DEVICE)

In [ ]:
#create dataset (training and validation)
ds = RNNDACLatentDataset(data_cfg, split="train", device="cpu")

loader = DataLoader(
    ds,
    batch_size=4,   # start small
    shuffle=True,
    num_workers=0   # keep 0 for notebooks / debugging
)


val_ds = RNNDACLatentDataset(validation_cfg, split="validation", device="cpu")
val_loader = DataLoader(val_ds, batch_size=params["val_batches"], shuffle=False, num_workers=0)


In [ ]:
# Just run one batch -------------------------------

In [ ]:
#grab one batch from the dataset
batch = next(iter(loader))
batch = {k: v.to(DEVICE) for k, v in batch.items()}

print(batch["latents"].shape)  # [B, T, n_q*8]
print(batch["targets"].shape)    # [B, T, n_q]
print(batch["cond"].shape)     # [B, T, p]

batch = {k: v.to(DEVICE) for k, v in batch.items()}

In [ ]:
# un a batch
out = model(
    latents=batch["latents"],                  # [B, T, 72]
    cond=batch["cond"],                        # [B, T, p]
    target_codes=batch["targets"],             # [B, T, n_q] for teacher forcing
    cascade_mode=params["cascade_mode"],
)



In [ ]:

loss_dict = model.compute_loss(out["logits_per_codebook"], batch["targets"])
loss = loss_dict["total_loss"]


In [ ]:
loss

In [ ]:
# Now run a short bunch of batches ----------------------------------------- 


In [ ]:
model.train()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)


grad_clip = 1.0

train_loss_history = []
val_loss_history = []

train_iter = iter(loader)
val_iter = iter(val_loader)

for step in range(params["n_steps"]):
    # --------------------------------------------------
    # Training step
    # --------------------------------------------------
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(loader)
        batch = next(train_iter)

    batch = {k: v.to(DEVICE) for k, v in batch.items()}

    model.train()

    out = model(
        latents=batch["latents"],
        cond=batch["cond"],
        target_codes=batch["targets"],
        cascade_mode=params["cascade_mode"],
    )

    loss_dict = model.compute_loss(
        out["logits_per_codebook"],
        batch["targets"]
    )

    loss = loss_dict["total_loss"]
    per_q_losses = loss_dict["per_codebook_losses"]

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    train_loss_history.append(loss.item())

    # TensorBoard: training total + per-codebook
    writer.add_scalar("Loss/train_total", loss.item(), step)
    for q, q_loss in enumerate(per_q_losses):
        writer.add_scalar(f"Loss_train/codebook_{q}", q_loss.item(), step)

    # Print training stats
    if step % params["print_every"] == 0:
        recent = train_loss_history[-params["print_every"]:]
        avg_train_loss = sum(recent) / len(recent)
        print(f"time: {datetime.now()}, Step {step:4d} | Train Loss: {loss.item():.4f} | Train Avg: {avg_train_loss:.4f}")

    if step > 0 and step % params["checkpoint_every"] == 0:
        ckpt_path = save_checkpoint(
            output_dir=checkpoint_dir,
            step=step,
            model=model,
            optimizer=optimizer,
            params=params,
            model_config=model_cfg,
            data_config=data_cfg,
            extra={
                "last_train_loss": loss.item(),
                "train_loss_history_tail": train_loss_history[-10:],
            },
        )
        
        print(f"Saved checkpoint: {ckpt_path}")
    # --------------------------------------------------
    # Validation
    # --------------------------------------------------
    if step % params["validate_every"] == 0:
        model.eval()

        val_total_losses = []
        val_per_q_sums = [0.0 for _ in range(model.config.n_q)]

        with torch.no_grad():
            for _ in range(params["val_batches"]):
                try:
                    val_batch = next(val_iter)
                except StopIteration:
                    val_iter = iter(val_loader)
                    val_batch = next(val_iter)

                val_batch = {k: v.to(DEVICE) for k, v in val_batch.items()}

                val_out = model(
                    latents=val_batch["latents"],
                    cond=val_batch["cond"],
                    target_codes=val_batch["targets"],
                    cascade_mode=params["cascade_mode"],
                )

                val_loss_dict = model.compute_loss(
                    val_out["logits_per_codebook"],
                    val_batch["targets"]
                )

                val_total_losses.append(val_loss_dict["total_loss"].item())

                for q, q_loss in enumerate(val_loss_dict["per_codebook_losses"]):
                    val_per_q_sums[q] += q_loss.item()

        avg_val_loss = sum(val_total_losses) / len(val_total_losses)
        avg_val_per_q = [x / params["val_batches"] for x in val_per_q_sums]

        val_loss_history.append((step, avg_val_loss))

        # TensorBoard: validation total + per-codebook
        writer.add_scalar("Loss/val_total", avg_val_loss, step)
        for q, q_loss in enumerate(avg_val_per_q):
            writer.add_scalar(f"Loss_val/codebook_{q}", q_loss, step)

        print(f"           | Val Avg ({params['val_batches']} batches): {avg_val_loss:.4f}")

        model.train()

writer.flush()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(train_loss_history, label="train loss")

if val_loss_history:
    val_steps, val_vals = zip(*val_loss_history)
    plt.plot(val_steps, val_vals, "o-", label="val loss")

plt.xlabel("step")
plt.ylabel("loss")
plt.title("Training and validation loss")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
####### Prepare for Inference, test configuration and model loading #############

In [ ]:
print(f'About to load {ckpt_path}')
saved_configs = torch.load(ckpt_path, weights_only=False)

model_config = GRUModelConfig(**saved_configs["model_config"])
data_config = LatentDatasetConfig(**saved_configs["data_config"])
params = saved_configs["params"]

In [ ]:
model = RNNDACModel(model_config, dac_model=dac_model)
ckpt = torch.load(ckpt_path, weights_only=False, map_location="cpu")
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

In [ ]:
# Actually do some inference (using the chunk+hopoe+lookahead strategy)

In [ ]:
listen_loader = DataLoader(val_ds, batch_size=1, shuffle=True, num_workers=0)
listen_batch = next(iter(listen_loader))
listen_batch = {k: v.to(DEVICE) for k, v in listen_batch.items()}

print(listen_batch["latents"].shape)   # [1, T, 72]
print(listen_batch["targets"].shape)   # [1, T, 9]
print(listen_batch["cond"].shape)      # [1, T, p]

In [ ]:
model.eval()

In [ ]:
from core.inference import infer_streaming_with_lookahead

#cvect = listen_batch["cond"]
cvect = torch.linspace(.1, .9, 300).unsqueeze(0).unsqueeze(-1)  # shape [1, 300, 1]

with torch.no_grad():
    gen_audio = infer_streaming_with_lookahead(
        rnn_model=model,
        dac_model=dac_model,
        cond_sequence=cvect,   # [1, T, p]
        chunk_size=16,
        hop_size=8,
        right_context=4,
        top_k=1,              # 1 = argmax
        temperature=1.0,
        frame_samples=512,
    )

print(f"conditioning shape: {cvect.shape}")
print(f"Audio shape: {gen_audio.shape}")
print(f" T*512 = {cvect.shape[1]*512}")

In [ ]:
import IPython.display as ipd

gen_audio_np = gen_audio.squeeze().detach().cpu().numpy()
ipd.display(ipd.Audio(gen_audio_np, rate=44100))